In [10]:
from IPython.terminal.shortcuts.auto_suggest import accept
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from langchain.tools import tool
from typing import Dict,Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def recipe_search(query:str) -> Dict[str, Any]:

    """ Search the web for the Food Recipe Information"""

    return tavily_client.search(query)

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
You are the Chef who has the recipes to make food with any Ingredients that is available.
Rules:
1. Give the recipe one by one in steps
2. Include timings of the steps(example :- cook on medium heat for 5 minutes, marinate for 30 minutes)
3. Only Use the Ingredients mentioned by the user
4. And if Ingredients are not mentioned then assume User has all the required Ingredients
5. If the User give image of the Ingredient then analyze the image and then list those ingredients
6. If the User give audio asking for recipe give the user the recipe
"""

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    system_prompt=system_prompt,
    tools = [recipe_search],
    checkpointer = InMemorySaver()
)

In [13]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm


duration = 5
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate),
               samplerate=sample_rate, channels=1)

for _ in tqdm(range(duration*10)):
    time.sleep(0.1)
sd.wait()
print("Done.")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.75it/s]


Done.


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

uploaded_file = uploader.value[0]

content_mv = uploaded_file["content"]

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [14]:
from langchain.messages import HumanMessage

try:
    img_b64
except NameError:
    img_b64 = None

try:
    aud_b64
except NameError:
    aud_b64 = None


while True:
    message_content = []
    user_text = input("Put your query here(or Press Enter for Image reading only)\n")
    if user_text.lower() == "quit":
        print("Chef : Goodbye")
        break

    if user_text:
        message_content.append({"type" : "text", "text" : user_text})

    if img_b64:
        message_content.append(
            {"type" : "image_url",
             "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
        )

    if aud_b64:
        message_content.append(
            {"type": "media", "mime_type": "audio/wav", "data": aud_b64}
        )

    if not message_content:
        print("Chef : You didnt give me any text or Ingredients")
        continue


    multimodal_question = HumanMessage(content=message_content)
    config = {"configurable": {"thread_id": "1"}}

    response = agent.invoke(
        {"messages" : [multimodal_question]},
        config
    )

    img_b64 = None
    aud_b64 = None

    print(response['messages'][-1].content)



[{'type': 'text', 'text': 'Here is a recipe for Chicken Biryani without pre-made biryani masala:\n\n**Ingredients:**\n*   Chicken\n*   Basmati Rice\n*   Onions\n*   Ginger-garlic paste\n*   Whole spices (bay leaf, cardamoms, cloves, cinnamon, star anise, shahi jeera, mace - use what you have)\n*   Oil or Ghee\n*   Salt\n*   Water\n\n**Instructions:**\n\n**Step 1: Prepare the Onions**\n*   Heat oil or ghee in a heavy-bottomed pan or Dutch oven over a medium flame.\n*   Add sliced onions and sauté until they turn golden to light brown. This will take approximately 14 minutes (7 minutes on medium heat, 4 minutes on low heat, and 2 to 3 minutes with the stove turned off).\n*   Remove the browned onions to a plate, leaving any excess ghee in the pan. Reserve about 2 tablespoons of these brown onions for later.\n\n**Step 2: Cook the Chicken**\n*   In the same pan with the remaining ghee, add the chicken.\n*   Sauté the chicken for 4 to 5 minutes on a medium heat until it turns pale.\n*   Cov